# DeBERTa-v3-base — Prompt-Based PCL Detection (BEIKE-NLP Style)

This notebook extends the baseline DeBERTa classifier with three improvements:

1. **Weighted Random Sampler** — sample weight = `1/√k_p` (PCL) or `1/√k_n` (No-PCL), where `k_p` / `k_n` are the class ratios in the training set.
2. **Structured input** — keyword and country are concatenated to the paragraph using special token boundary pairs `<keyword>…</keyword>` and `<country>…</country>`.
3. **Prompt classification (BEIKE-NLP)** — following the cloze-prompt paradigm: the task is reformulated as a masked-token fill-in problem. The input is wrapped in a natural-language template, a `[MASK]` token acts as the label slot, and a **verbalizer** maps label words in the vocabulary to the two classes. The DeBERTa MLM head predicts over the verbalizer words at the `[MASK]` position rather than using a linear classification head.

**Prompt template:**
```
<keyword> {keyword} </keyword> <country> {country} </country> {text} The tone of this passage is [MASK].
```
**Verbalizer:** `neutral` → Class 0 (No PCL) · `patronizing` → Class 1 (PCL)

In [ ]:
# ============================================================
# Cell 1: Install dependencies (run once, then restart kernel)
# ============================================================
# !pip install torch transformers contractions sentencepiece protobuf scikit-learn

In [ ]:
# ============================================================
# Cell 2: Imports and reproducibility seeds
# ============================================================
import os
import re
import sys
import random
import logging
import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import contractions

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForMaskedLM, get_cosine_schedule_with_warmup

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)

# Add parent directory to path so dont_patronize_me.py is importable
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
from dont_patronize_me import DontPatronizeMe


# ---------- Fix all seeds ----------
SEED = 42

def set_seed(seed: int = 42) -> None:
    """Seed Python, NumPy, PyTorch (CPU + CUDA) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

In [ ]:
# ============================================================
# Cell 3: Hyper-parameters and paths
# ============================================================

# Model
MODEL_NAME    = 'microsoft/deberta-v3-base'
MAX_LENGTH    = 256
DROPOUT_RATE  = 0.4
NUM_LABELS    = 2

# Verbalizer: maps class index → label word fed to the MLM head
# The MLM head predicts over vocabulary tokens at the [MASK] position;
# only the logits for these two words are used for classification.
VERBALIZER = {
    0: 'neutral',      # No-PCL
    1: 'patronizing',  # PCL
}

# Prompt template (Python format string).
# {keyword}, {country}, {text} are filled at dataset creation time.
# The mask placeholder is replaced with tokenizer.mask_token at runtime.
PROMPT_TEMPLATE = (
    '<keyword> {keyword} </keyword> '
    '<country> {country} </country> '
    '{text} '
    'The tone of this passage is {mask}.'
)

# Tokens reserved for the non-text parts of the prompt
# (special boundary tokens + template words + [CLS]/[SEP]).
# Set conservatively so the text is never truncated past the [MASK].
PROMPT_TOKEN_OVERHEAD = 40

# Training
BATCH_SIZE    = 16
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.10
MAX_GRAD_NORM = 1.0

# Early stopping
EVAL_EVERY    = 1    # evaluate dev set every N training batches
PATIENCE      = 50  # training batches without improvement → stop

# Logging / checkpointing
LOG_EVERY      = 50
CHECKPOINT_DIR = os.path.join('..', 'checkpoints', 'deberta_v3_prompt_pcl')
LOG_DIR        = os.path.join('..', 'logs')
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print('Configuration loaded.')

In [ ]:
# ============================================================
# Cell 4: Logging setup
# ============================================================

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_file  = os.path.join(LOG_DIR, f'prompt_training_{timestamp}.log')

for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(log_file, mode='w'),
    ],
)
logger = logging.getLogger('pcl_prompt_train')

logger.info(f'Log file : {log_file}')
logger.info(f'Device   : {DEVICE}')
logger.info(f'Model    : {MODEL_NAME}')

In [ ]:
# ============================================================
# Cell 5: Load raw dataset
# ============================================================

DATA_ROOT = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..')

dpm = DontPatronizeMe(train_path=DATA_ROOT, test_path=os.path.join(DATA_ROOT, 'test', 'task4_test.tsv'))
dpm.load_task1()
df = dpm.train_task1_df

df['par_id'] = df['par_id'].astype(str)
df['label']  = df['label'].astype(int)

logger.info(f'Total samples: {len(df):,}')
logger.info(f'Class distribution:\n{df["label"].value_counts().sort_index().to_string()}')
print(df[['par_id', 'keyword', 'country', 'text', 'label']].head(3))

In [ ]:
# ============================================================
# Cell 6: Text preprocessing
# ============================================================

def preprocess_text(text: str) -> str:
    """
    1. Expand contractions  (you've → you have)
    2. Remove HTML tags     (<h> ... </h>)
    3. Keep only alphabetic characters and whitespace
    4. Normalise whitespace
    Case is preserved — DeBERTa-v3-base is a cased model.
    """
    text = str(text)
    text = contractions.fix(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Sanity check
samples = [
    "You've got to help these <b>vulnerable</b> people!",
    "It's & it can't be denied. <h> Headline </h>",
]
for s in samples:
    print(repr(s), '->', repr(preprocess_text(s)))

df['clean_text'] = df['text'].apply(preprocess_text)
logger.info('Text preprocessing complete.')

In [ ]:
# ============================================================
# Cell 7: Official train / dev split + class-ratio statistics
# ============================================================

train_ids_df = pd.read_csv(os.path.join(DATA_ROOT, 'train', 'train_semeval_parids-labels.csv'))
dev_ids_df   = pd.read_csv(os.path.join(DATA_ROOT, 'train', 'dev_semeval_parids-labels.csv'))

train_par_ids = set(train_ids_df['par_id'].astype(str))
dev_par_ids   = set(dev_ids_df['par_id'].astype(str))

train_df = df[df['par_id'].isin(train_par_ids)].reset_index(drop=True)
dev_df   = df[df['par_id'].isin(dev_par_ids)].reset_index(drop=True)

leftover_df = df[~df['par_id'].isin(train_par_ids | dev_par_ids)].reset_index(drop=True)
if len(leftover_df) > 0:
    logger.warning(f'{len(leftover_df)} unassigned samples appended to training set.')
    train_df = pd.concat([train_df, leftover_df], ignore_index=True)

# ── Class ratios used for the WeightedRandomSampler ──────────────────────────
# k_p = proportion of positive (PCL=1) examples in training set
# k_n = proportion of negative (No-PCL=0) examples in training set
n_train  = len(train_df)
n_pcl    = (train_df['label'] == 1).sum()
n_no_pcl = (train_df['label'] == 0).sum()

k_p = n_pcl    / n_train   # ~0.095
k_n = n_no_pcl / n_train   # ~0.905

logger.info(f'Train size : {n_train:,}  (PCL={n_pcl:,}, No-PCL={n_no_pcl:,})')
logger.info(f'k_p = {k_p:.4f}  |  k_n = {k_n:.4f}')
logger.info(f'Sample weight for PCL    : 1/√k_p = {1/np.sqrt(k_p):.4f}')
logger.info(f'Sample weight for No-PCL : 1/√k_n = {1/np.sqrt(k_n):.4f}')

print(f'Train: {n_train:,}  |  Dev: {len(dev_df):,}')
print(f'k_p={k_p:.4f}  k_n={k_n:.4f}')
print(f'w_pcl={1/np.sqrt(k_p):.4f}  w_no_pcl={1/np.sqrt(k_n):.4f}')

In [ ]:
# ============================================================
# Cell 8: Tokeniser, special boundary tokens, and verbalizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── 1. Register special boundary tokens ──────────────────────────────────────
# These mark keyword and country fields inside the prompt so the model can
# learn to distinguish them from the paragraph text.
BOUNDARY_TOKENS = ['<keyword>', '</keyword>', '<country>', '</country>']
num_added = tokenizer.add_special_tokens({'additional_special_tokens': BOUNDARY_TOKENS})
logger.info(f'Added {num_added} special boundary tokens to tokenizer.')

MASK_TOKEN    = tokenizer.mask_token       # '[MASK]' for DeBERTa
MASK_TOKEN_ID = tokenizer.mask_token_id
print(f'Mask token: {MASK_TOKEN!r}  (id={MASK_TOKEN_ID})')

# ── 2. Resolve verbalizer token IDs ──────────────────────────────────────────
# Each label word is encoded with a leading space (SentencePiece convention
# treats words mid-sentence with a ▁ prefix).  If a label word is split into
# multiple sub-tokens, only the first sub-token ID is used — this is standard
# practice in PET-style prompt classification.

def resolve_verbalizer_ids(tokenizer, verbalizer: dict) -> list:
    """Return a list of token IDs in class-index order."""
    ids = []
    for label_idx in sorted(verbalizer.keys()):
        word   = verbalizer[label_idx]
        # Encode with a space prefix to match mid-sentence tokenisation
        tokens = tokenizer.encode(' ' + word, add_special_tokens=False)
        tok_id = tokens[0]  # first sub-token
        n_toks = len(tokens)
        status = 'single-token' if n_toks == 1 else f'multi-token ({n_toks} pieces, using first)'
        logger.info(f'  Class {label_idx} → "{word}" → token id {tok_id}  [{status}]')
        print(f'  Class {label_idx} → "{word}" → token id {tok_id}  [{status}]')
        ids.append(tok_id)
    return ids

print('\nVerbalizer token IDs:')
VERBALIZER_IDS = resolve_verbalizer_ids(tokenizer, VERBALIZER)

In [ ]:
# ============================================================
# Cell 9: Prompt builder
# ============================================================

def build_prompt(text: str, keyword: str, country: str, mask_token: str) -> str:
    """
    Fill PROMPT_TEMPLATE with the preprocessed text, keyword, country,
    and the tokenizer's mask token.

    The text must already be pre-truncated so the full prompt fits within
    MAX_LENGTH tokens (see PCLPromptDataset.__getitem__).
    """
    return PROMPT_TEMPLATE.format(
        keyword=keyword,
        country=country,
        text=text,
        mask=mask_token,
    )

# Smoke test
demo_text    = 'These poor children really need our help'
demo_prompt  = build_prompt(demo_text, 'homeless', 'gb', MASK_TOKEN)
print('Example prompt:')
print(demo_prompt)
print()
demo_enc = tokenizer(demo_prompt, truncation=False)
print(f'Tokens: {len(demo_enc["input_ids"])}')

In [ ]:
# ============================================================
# Cell 10: Dataset
# ============================================================

class PCLPromptDataset(Dataset):
    """
    Each sample is a cloze-style prompt:

        <keyword> {kw} </keyword> <country> {cc} </country>
        {paragraph text}  The tone of this passage is [MASK].

    The dataset returns:
        input_ids, attention_mask  — standard tokenizer outputs
        mask_pos                   — position of [MASK] in input_ids
        label                      — binary ground-truth (0 / 1)
    """

    def __init__(
        self,
        dataframe: pd.DataFrame,
        tokenizer,
        max_length: int = MAX_LENGTH,
        prompt_overhead: int = PROMPT_TOKEN_OVERHEAD,
    ) -> None:
        self.df             = dataframe.reset_index(drop=True)
        self.tokenizer      = tokenizer
        self.max_length     = max_length
        self.mask_token     = tokenizer.mask_token
        self.mask_token_id  = tokenizer.mask_token_id
        # Maximum sub-word tokens the paragraph text may occupy
        self.text_max_tokens = max_length - prompt_overhead

    def __len__(self) -> int:
        return len(self.df)

    def _truncate_text(self, text: str) -> str:
        """Truncate text to at most self.text_max_tokens sub-word tokens."""
        token_ids     = self.tokenizer.encode(text, add_special_tokens=False)
        token_ids     = token_ids[:self.text_max_tokens]
        return self.tokenizer.decode(token_ids, skip_special_tokens=True)

    def __getitem__(self, idx: int) -> dict:
        row          = self.df.iloc[idx]
        trunc_text   = self._truncate_text(row['clean_text'])
        prompt       = build_prompt(trunc_text, row['keyword'], row['country'], self.mask_token)

        encoding = self.tokenizer(
            prompt,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        input_ids      = encoding['input_ids'].squeeze(0)       # (seq_len,)
        attention_mask = encoding['attention_mask'].squeeze(0)  # (seq_len,)

        # Find the [MASK] position in the padded sequence
        mask_positions = (input_ids == self.mask_token_id).nonzero(as_tuple=True)[0]
        if len(mask_positions) > 0:
            mask_pos = mask_positions[0].item()
        else:
            # Fallback: [MASK] was truncated — use last non-padding token
            mask_pos = int(attention_mask.sum().item()) - 1

        return {
            'input_ids'     : input_ids,
            'attention_mask': attention_mask,
            'mask_pos'      : torch.tensor(mask_pos, dtype=torch.long),
            'label'         : torch.tensor(row['label'], dtype=torch.long),
        }


train_dataset = PCLPromptDataset(train_df, tokenizer)
dev_dataset   = PCLPromptDataset(dev_df,   tokenizer)

# Verify that [MASK] always lands within the sequence
sample = train_dataset[0]
print(f'Sample input_ids shape  : {sample["input_ids"].shape}')
print(f'[MASK] position         : {sample["mask_pos"].item()}')
print(f'Token at mask_pos       : {tokenizer.convert_ids_to_tokens([sample["input_ids"][sample["mask_pos"]].item()])}')
print(f'Label                   : {sample["label"].item()}')

In [ ]:
# ============================================================
# Cell 11: DataLoaders with Weighted Random Sampler
# ============================================================
#
# Per-sample weight:
#   w_i = 1 / √k_p   if sample i is PCL  (label=1)
#   w_i = 1 / √k_n   if sample i is No-PCL (label=0)
#
# These weights reduce (but do not fully eliminate) the ~10:1 class imbalance
# by upsampling the minority class with a softer correction than 1/k.

w_pcl    = 1.0 / np.sqrt(k_p)   # weight for positive (PCL) samples
w_no_pcl = 1.0 / np.sqrt(k_n)   # weight for negative (No-PCL) samples

sample_weights = np.where(
    train_df['label'].values == 1,
    w_pcl,
    w_no_pcl,
)

sampler = WeightedRandomSampler(
    weights     = torch.from_numpy(sample_weights).double(),
    num_samples = len(sample_weights),
    replacement = True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    sampler     = sampler,          # mutually exclusive with shuffle=True
    num_workers = 0,
    pin_memory  = (DEVICE.type == 'cuda'),
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size  = BATCH_SIZE * 2,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = (DEVICE.type == 'cuda'),
)

logger.info(f'w_pcl={w_pcl:.4f}  w_no_pcl={w_no_pcl:.4f}')
logger.info(f'Train batches/epoch: {len(train_loader):,}')

print(f'Train batches / epoch : {len(train_loader):,}')
print(f'Dev   batches         : {len(dev_loader):,}')
print(f'Sampling weights      :  PCL={w_pcl:.4f}  |  No-PCL={w_no_pcl:.4f}')

In [ ]:
# ============================================================
# Cell 12: Model — DeBERTa MLM head with prompt classification
# ============================================================
#
# Architecture (BEIKE-NLP cloze-prompt style):
#
#   DeBERTa-v3-base (AutoModelForMaskedLM)
#       └─ full-sequence hidden states  (batch, seq_len, hidden_size)
#           └─ MLM projection head      (batch, seq_len, vocab_size)
#               └─ logits at [MASK] pos (batch, vocab_size)
#                   └─ slice verbalizer (batch, 2)      ← 2 label words
#                       └─ Softmax(dim=-1)              ← class probs
#
# The model is trained end-to-end with CrossEntropyLoss on the 2-way
# verbalizer logits (no separate classification head is added).

class DeBERTaPromptClassifier(nn.Module):

    def __init__(
        self,
        model_name    : str  = MODEL_NAME,
        verbalizer_ids: list = None,
    ) -> None:
        super().__init__()
        # MLM model (includes the language-model head on top of DeBERTa encoder)
        self.mlm            = AutoModelForMaskedLM.from_pretrained(model_name)
        self.verbalizer_ids = verbalizer_ids  # [id_class0, id_class1]
        self.softmax        = nn.Softmax(dim=-1)

    def resize_token_embeddings(self, new_num_tokens: int) -> None:
        """Grow embedding table after adding special tokens to the tokenizer."""
        self.mlm.resize_token_embeddings(new_num_tokens)

    def forward(
        self,
        input_ids     : torch.Tensor,  # (batch, seq_len)
        attention_mask: torch.Tensor,  # (batch, seq_len)
        mask_pos      : torch.Tensor,  # (batch,)  — index of [MASK] in each sequence
        return_probs  : bool = False,
    ) -> torch.Tensor:
        # Full-vocab logits for every token position
        outputs    = self.mlm(input_ids=input_ids, attention_mask=attention_mask)
        all_logits = outputs.logits                     # (batch, seq_len, vocab_size)

        # Gather logits only at the [MASK] position for each example
        batch_size = input_ids.shape[0]
        batch_idx  = torch.arange(batch_size, device=input_ids.device)
        mask_logits = all_logits[batch_idx, mask_pos, :]  # (batch, vocab_size)

        # Project down to verbalizer vocabulary (2 words → 2 logits)
        verb_ids    = torch.tensor(self.verbalizer_ids, device=input_ids.device)
        verb_logits = mask_logits[:, verb_ids]            # (batch, 2)

        if return_probs:
            return self.softmax(verb_logits)
        return verb_logits


model = DeBERTaPromptClassifier(verbalizer_ids=VERBALIZER_IDS)

# Resize embeddings to include the 4 new boundary tokens
model.resize_token_embeddings(len(tokenizer))
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
logger.info(f'Model loaded          : {MODEL_NAME}')
logger.info(f'Trainable parameters  : {n_params:,}')
logger.info(f'Verbalizer token IDs  : {VERBALIZER_IDS}')
print(f'Trainable parameters: {n_params:,}')

In [ ]:
# ============================================================
# Cell 13: Optimizer, scheduler, and loss
# ============================================================
#
# Loss: standard CrossEntropyLoss (no class weights).
# The WeightedRandomSampler already adjusts for class imbalance by
# up-sampling PCL examples; adding class-weighted loss would
# double-correct and can destabilise training.

optimizer = AdamW(
    model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

criterion = nn.CrossEntropyLoss()

logger.info(f'Total steps : {total_steps:,}')
logger.info(f'Warmup steps: {warmup_steps:,}  ({WARMUP_RATIO*100:.0f}%)')
logger.info(f'Optimizer   : AdamW  lr={LEARNING_RATE}  wd={WEIGHT_DECAY}')
logger.info(f'Scheduler   : CosineAnnealingWithLinearWarmup')
logger.info(f'Loss        : CrossEntropyLoss (no class weights — WRS handles imbalance)')

print(f'Total steps: {total_steps:,}  |  Warmup: {warmup_steps:,}')

In [ ]:
# ============================================================
# Cell 14: Evaluation helper
# ============================================================

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """Run inference on a DataLoader and return per-class and macro metrics."""
    model.eval()
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        mask_pos       = batch['mask_pos'].to(DEVICE)
        labels         = batch['label'].cpu().numpy()

        probs = model(input_ids, attention_mask, mask_pos, return_probs=True)
        preds = probs.argmax(dim=-1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    return {
        'f1_macro'    : f1_score(all_labels, all_preds, average='macro',    zero_division=0),
        'f1_no_pcl'   : f1_score(all_labels, all_preds, pos_label=0, average='binary', zero_division=0),
        'f1_pcl'      : f1_score(all_labels, all_preds, pos_label=1, average='binary', zero_division=0),
        'prec_no_pcl' : precision_score(all_labels, all_preds, pos_label=0, average='binary', zero_division=0),
        'prec_pcl'    : precision_score(all_labels, all_preds, pos_label=1, average='binary', zero_division=0),
        'rec_no_pcl'  : recall_score(all_labels, all_preds,  pos_label=0, average='binary', zero_division=0),
        'rec_pcl'     : recall_score(all_labels, all_preds,  pos_label=1, average='binary', zero_division=0),
        'preds'       : all_preds,
        'labels'      : all_labels,
    }

In [ ]:
# ============================================================
# Cell 15: Training loop
# ============================================================

best_val_f1      = -1.0
no_improve_count = 0
global_step      = 0
early_stopped    = False

history = {
    'train_loss'  : [],   # (global_step, loss)
    'val_f1_macro': [],   # (global_step, f1_macro)
    'val_f1_pcl'  : [],   # (global_step, f1_pcl)
}

logger.info('=' * 60)
logger.info('Training started  (prompt-based)')
logger.info('=' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    if early_stopped:
        break

    model.train()
    epoch_loss  = 0.0
    epoch_steps = 0

    for batch_idx, batch in enumerate(train_loader, start=1):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        mask_pos       = batch['mask_pos'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        # ── Forward + backward ──────────────────────────────────
        optimizer.zero_grad()
        verb_logits = model(input_ids, attention_mask, mask_pos)  # (batch, 2)
        loss        = criterion(verb_logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()

        epoch_loss  += loss.item()
        epoch_steps += 1
        global_step += 1

        # ── Periodic loss logging ────────────────────────────────
        if global_step % LOG_EVERY == 0:
            logger.info(
                f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
                f'Batch {batch_idx:04d}/{len(train_loader)} | '
                f'Step {global_step:05d} | '
                f'LR {scheduler.get_last_lr()[0]:.2e} | '
                f'Loss {loss.item():.4f}'
            )

        history['train_loss'].append((global_step, loss.item()))

        # ── Evaluation every EVAL_EVERY batches ─────────────────
        if global_step % EVAL_EVERY == 0:
            val_metrics = evaluate(model, dev_loader)
            val_f1_mac  = val_metrics['f1_macro']
            val_f1_pcl  = val_metrics['f1_pcl']

            history['val_f1_macro'].append((global_step, val_f1_mac))
            history['val_f1_pcl'].append((global_step, val_f1_pcl))

            if val_f1_pcl > best_val_f1:
                best_val_f1      = val_f1_pcl
                no_improve_count = 0

                torch.save(
                    {
                        'epoch'               : epoch,
                        'global_step'         : global_step,
                        'model_state_dict'    : model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'best_val_f1'         : best_val_f1,
                        'verbalizer_ids'      : VERBALIZER_IDS,
                        'verbalizer'          : VERBALIZER,
                        'config': {
                            'model_name'     : MODEL_NAME,
                            'max_length'     : MAX_LENGTH,
                            'prompt_template': PROMPT_TEMPLATE,
                        },
                    },
                    BEST_CKPT_PATH,
                )
                logger.info(
                    f'  ✓ New best  PCL-F1={best_val_f1:.4f}  '
                    f'macro-F1={val_f1_mac:.4f}  '
                    f'[step {global_step}]  → checkpoint saved'
                )
            else:
                no_improve_count += 1

            # ── Early stopping ───────────────────────────────────
            if no_improve_count >= PATIENCE:
                logger.info(
                    f'Early stopping triggered at step {global_step} '
                    f'(no improvement for {PATIENCE} consecutive batches).'
                )
                early_stopped = True
                break

            model.train()  # restore train mode after eval

    avg_loss = epoch_loss / max(epoch_steps, 1)
    logger.info(
        f'--- Epoch {epoch:02d}/{NUM_EPOCHS} | '
        f'Avg train loss: {avg_loss:.4f} | '
        f'Best PCL-F1: {best_val_f1:.4f} ---'
    )

logger.info('=' * 60)
logger.info(f'Training finished.  Best dev PCL-class F1: {best_val_f1:.4f}')
logger.info(f'Best checkpoint: {BEST_CKPT_PATH}')
logger.info('=' * 60)

In [ ]:
# ============================================================
# Cell 16: Training curves
# ============================================================

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Training loss
if history['train_loss']:
    steps, losses = zip(*history['train_loss'])
    axes[0].plot(steps, losses, linewidth=0.6, alpha=0.4, color='#4C72B0')
    window   = min(50, len(losses))
    smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
    axes[0].plot(steps[window - 1:], smoothed, linewidth=1.8, color='#003d80', label='Smoothed')
    axes[0].set_title('(a) Training Loss')
    axes[0].set_xlabel('Global Step')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].legend(fontsize=9)

# (b) Dev macro-F1
if history['val_f1_macro']:
    steps, f1s = zip(*history['val_f1_macro'])
    axes[1].plot(steps, f1s, color='#55A868', linewidth=1.6, marker='o', markersize=2)
    best_macro = max(f1s)
    axes[1].axhline(best_macro, linestyle='--', color='grey', linewidth=1,
                    label=f'Best={best_macro:.4f}')
    axes[1].set_title('(b) Dev Macro-F1')
    axes[1].set_xlabel('Global Step')
    axes[1].set_ylabel('Macro F1')
    axes[1].legend(fontsize=9)

# (c) Dev PCL-class F1
if history['val_f1_pcl']:
    steps, f1s = zip(*history['val_f1_pcl'])
    best_pcl   = max(f1s)
    axes[2].plot(steps, f1s, color='#DD8452', linewidth=1.6, marker='o', markersize=2)
    axes[2].axhline(best_pcl, linestyle='--', color='grey', linewidth=1,
                    label=f'Best={best_pcl:.4f}')
    axes[2].set_title('(c) Dev PCL-Class F1')
    axes[2].set_xlabel('Global Step')
    axes[2].set_ylabel('F1 (PCL class)')
    axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'prompt_training_curves_{timestamp}.png'), bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 17: Load best checkpoint
# ============================================================

checkpoint = torch.load(BEST_CKPT_PATH, map_location=DEVICE)

best_model = DeBERTaPromptClassifier(
    model_name     = MODEL_NAME,
    verbalizer_ids = checkpoint['verbalizer_ids'],
)
best_model.resize_token_embeddings(len(tokenizer))
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model = best_model.to(DEVICE)
best_model.eval()

logger.info(
    f'Loaded best checkpoint from epoch {checkpoint["epoch"]}, '
    f'step {checkpoint["global_step"]}, '
    f'dev PCL-F1={checkpoint["best_val_f1"]:.4f}'
)
print(f'Best checkpoint: epoch {checkpoint["epoch"]} | '
      f'step {checkpoint["global_step"]} | '
      f'val PCL-F1={checkpoint["best_val_f1"]:.4f}')

---
## Evaluation Harness

Evaluates the best checkpoint on the held-out dev set and reports per-class **F1**, **Precision**, and **Recall** for both `No PCL (0)` and `PCL (1)` classes, plus a full `sklearn` classification report, confusion matrix, and bar-chart visualisation.

In [ ]:
# ============================================================
# Cell 18: Run evaluation and print metrics
# ============================================================

eval_metrics = evaluate(best_model, dev_loader)

print('=' * 60)
print('EVALUATION RESULTS — Dev Set  (prompt-based DeBERTa-v3)')
print('=' * 60)

results_table = pd.DataFrame({
    'Class'    : ['No PCL (0)', 'PCL (1)'],
    'Precision': [eval_metrics['prec_no_pcl'], eval_metrics['prec_pcl']],
    'Recall'   : [eval_metrics['rec_no_pcl'],  eval_metrics['rec_pcl']],
    'F1'       : [eval_metrics['f1_no_pcl'],   eval_metrics['f1_pcl']],
})
results_table[['Precision', 'Recall', 'F1']] = (
    results_table[['Precision', 'Recall', 'F1']].round(4)
)

print(f'\nMacro-F1 : {eval_metrics["f1_macro"]:.4f}')
print()
print(results_table.to_string(index=False))
print('\n--- Full Classification Report ---')
print(
    classification_report(
        eval_metrics['labels'],
        eval_metrics['preds'],
        target_names=['No PCL (0)', 'PCL (1)'],
        digits=4,
    )
)

logger.info(f'Dev macro-F1  : {eval_metrics["f1_macro"]:.4f}')
logger.info(f'Dev F1 (PCL)  : {eval_metrics["f1_pcl"]:.4f}')
logger.info(f'Dev Prec (PCL): {eval_metrics["prec_pcl"]:.4f}')
logger.info(f'Dev Rec  (PCL): {eval_metrics["rec_pcl"]:.4f}')

In [ ]:
# ============================================================
# Cell 19: Evaluation plots
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_cls = ['#4C72B0', '#DD8452']

# ── (a) Per-class Precision / Recall / F1 ─────────────────────────────────
ax = axes[0]
metrics_names = ['Precision', 'Recall', 'F1']
no_pcl_vals   = [eval_metrics['prec_no_pcl'], eval_metrics['rec_no_pcl'], eval_metrics['f1_no_pcl']]
pcl_vals      = [eval_metrics['prec_pcl'],    eval_metrics['rec_pcl'],    eval_metrics['f1_pcl']]

x  = np.arange(len(metrics_names))
w  = 0.30
b1 = ax.bar(x - w / 2, no_pcl_vals, width=w, color=colors_cls[0],
            label='No PCL (0)', edgecolor='white')
b2 = ax.bar(x + w / 2, pcl_vals,    width=w, color=colors_cls[1],
            label='PCL (1)',    edgecolor='white')

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2, h + 0.01,
        f'{h:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold',
    )

ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('(a) Precision / Recall / F1 per Class')
ax.axhline(eval_metrics['f1_macro'], linestyle='--', color='grey',
           linewidth=1.2, label=f'Macro-F1={eval_metrics["f1_macro"]:.3f}')
ax.legend(fontsize=9)

# ── (b) Confusion matrix ──────────────────────────────────────────────────
ax  = axes[1]
cm  = confusion_matrix(eval_metrics['labels'], eval_metrics['preds'])
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred: No PCL', 'Pred: PCL'],
    yticklabels=['True: No PCL', 'True: PCL'],
    ax=ax, linewidths=0.5, annot_kws={'size': 13},
)
ax.set_title('(b) Confusion Matrix')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, f'prompt_eval_metrics_{timestamp}.png'), bbox_inches='tight')
plt.show()

print(f'Evaluation plots saved to {LOG_DIR}/')